# 15. Build the final report

Regenerate the English LaTeX report strictly from persisted pipeline outputs, and show which files the renderer consumes so that every reported number is traceable to a CSV or JSON artefact.

**Reads**

- `outputs/tables/*.csv`
- `outputs/metrics/analysis_summary.json`
- `data/processed/*.csv`

**Writes**

- `report/report.tex`

**Method reference:** `METHODOLOGY.md` section 16

In [1]:
"""Notebook environment: locate the repository and expose its data layers."""

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve the repository root from wherever the kernel was started, so the
# notebook works both from the repository root and from the notebooks directory.
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
METRICS = ROOT / 'outputs' / 'metrics'

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

print('repository:', ROOT.name)
print('pipeline outputs present:', (PROCESSED / 'fiscal_balances_1977_2025.csv').exists())

repository: portugal-fiscal-balance
pipeline outputs present: True


## 1. Render

The renderer reads persisted outputs and writes LaTeX. It contains no analysis:
if a number is in the report, it is in a CSV or JSON file first. That is what
makes the report checkable without running any code.

In [2]:
from portugal_fiscal_balance.reporting.render import render_report

report_path = render_report(ROOT)
text = report_path.read_text(encoding='utf-8')
print('written:', report_path.relative_to(ROOT))
print('size:', f'{len(text) / 1024:.1f} kB')
print('lines:', len(text.splitlines()))

written: report\report.tex
size: 13.1 kB
lines: 322


## 2. Inputs the report consumes

In [3]:
consumed = pd.DataFrame(
    [
        {
            'artefact': str(path.relative_to(ROOT)).replace('\\', '/'),
            'size_kb': round(path.stat().st_size / 1024, 1),
        }
        for path in [
            *sorted((ROOT / 'outputs' / 'tables').glob('*.csv')),
            *sorted((ROOT / 'outputs' / 'metrics').glob('*.json')),
        ]
    ]
)
display(consumed)

,artefact,size_kb
0,outputs/tables/account_identity_checks.csv,13.3
1,outputs/tables/balance_change_attribution.csv,5.8
2,outputs/tables/debt_stock_flow_reconciliation.csv,11.8
3,outputs/tables/historical_ssf_labour_comovemen...,0.3
4,outputs/tables/historical_transfer_reallocatio...,9.9
5,outputs/tables/investment_diagnostic.csv,19.9
6,outputs/tables/nominal_gdp_balance_comovement.csv,0.5
7,outputs/tables/persistence_summary.csv,0.4
8,outputs/tables/primary_balance_and_interest.csv,23.2
9,outputs/tables/recent_balance_decomposition_20...,1.0


## 3. Structure of the generated report

In [4]:
sections = [line.strip() for line in text.splitlines() if line.startswith('\\section')]
print(len(sections), 'sections')
for line in sections:
    print(' -', line.removeprefix('\\section{').removesuffix('}'))

14 sections
 - \section*{Abstract
 - Data and Reproducibility
 - Long-Run Subsector Decomposition
 - Year-to-Year Attribution
 - Revenue and Expenditure Dynamics
 - Social Security Funds: Revenue Composition and Internal Systems
 - Primary Balance and Interest
 - Fixed-Capital Formation Diagnostic
 - Debt and Stock-Flow Adjustment
 - Persistence and Structural Mean Shifts
 - Descriptive Macroeconomic Co-Movement
 - Intergovernmental Transfers
 - Methodological Limitations
 - Reproducibility


In [5]:
figure_lines = [line.strip() for line in text.splitlines() if 'includegraphics' in line]
print(len(figure_lines), 'figures included from outputs/figures')
for line in figure_lines:
    print(' -', line.split('{')[-1].removesuffix('}'))

7 figures included from outputs/figures
 - 01_long_run_balances.png
 - 02_ssf_offset_ratio.png
 - 03_balance_change_attribution.png
 - 04_central_revenue_expenditure.png
 - 05_ssf_contribution_share.png
 - 06_central_primary_balance.png
 - 07_general_government_debt.png


## 4. First page of the source

In [6]:
print(text[:2500])

\documentclass[11pt]{article}
\usepackage[utf8]{inputenc}
\usepackage[T1]{fontenc}
\usepackage{amsmath}
\usepackage{booktabs}
\usepackage{graphicx}
\usepackage{geometry}
\usepackage{hyperref}
\geometry{margin=1in}
\graphicspath{{../outputs/figures/}}

\title{Portugal's General-Government Balance by Subsector, 1977--2025}
\author{Diogo Ribeiro}
\date{}

\begin{document}
\maketitle

\section*{Abstract}

This report decomposes Portugal's annual general-government net lending (+) / net borrowing (-) into Central Government, Regional and Local Government, and Social Security Funds (SSF). The analysis is empirical and accounting-focused. It studies long-run balance composition, year-to-year attribution, revenue and expenditure dynamics, persistence, structural mean shifts, Social Security revenue composition, primary balances, public investment, debt-flow reconciliation, and descriptive macroeconomic co-movement.

The central accounting identity is

\[
B^{GG}_t = B^{C}_t + B^{RL}_t + B^{SSF}

## Interpretation limits

1. The report **restates persisted results**. It introduces no new calculation
   and no conclusion that is not supported by an artefact in `outputs/`.
2. It carries the **same caveats** as the notebooks: the 1995 splice, the
   1996-1999 component gap, and the descriptive nature of every regression.
3. Regenerating the report without first running the pipeline reproduces the
   **previous** outputs, because the renderer reads files rather than recomputing
   them.

---

[Previous: 14. Descriptive macroeconomic co-movement](14_macroeconomic_comovement.ipynb)

Every table shown above is also persisted as CSV, so results can be checked without reading notebook state. To rebuild everything from the bundled raw sources:

```bash
poetry install
make all
```